# Análisis System Usability Scale (SUS) — SGB-SaaS

Cuaderno de análisis estadístico para las pruebas de usabilidad SUS (dataset retirado — N=0 en todo el entregable, ver OBS-08; este cuaderno conserva el pipeline validado sobre datos mock).

**Referencias:**
- Brooke, J. (1996). SUS: A "quick and dirty" usability scale.
- Bangor, A., Kortum, P. & Miller, J. (2008). An empirical evaluation of the System Usability Scale.
- Sauro, J. (2011). A practical guide to the System Usability Scale.

## 1. Configuración y carga de datos

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='colorblind')

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == 'scripts':
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

SUS_CSV = PROJECT_ROOT / 'docs' / 'mediciones' / 'sus' / 'sus.csv'
OUTPUT_DIR = PROJECT_ROOT / 'docs' / 'mediciones' / 'sus'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    print(f'SUS_CSV: {SUS_CSV.relative_to(PROJECT_ROOT)}')
except ValueError:
    print(f'SUS_CSV: {SUS_CSV}')
print(f'Existe: {SUS_CSV.exists()}')

SUS_CSV: docs\mediciones\sus\sus.csv
Existe: True


In [ ]:
df = pd.read_csv(SUS_CSV)
print(f'Datos cargados: N={len(df)} participantes')
print(f'Columnas: {list(df.columns)}')
df.head(15)

## 2. Verificación del cálculo SUS

Fórmula de Brooke (1996):
- Ítems **impares** (Q1, Q3, Q5, Q7, Q9): contribución = `valor - 1`
- Ítems **pares** (Q2, Q4, Q6, Q8, Q10): contribución = `5 - valor`
- Score = `(suma de contribuciones) × 2.5` → rango 0–100

In [3]:
ITEM_COLS = [f'Q{i}' for i in range(1, 11)]

def calcular_score_sus(row):
    suma = 0
    for i in range(10):
        val = int(row[f'Q{i+1}'])
        if i % 2 == 0:  # impar (Q1,Q3,Q5,Q7,Q9)
            suma += val - 1
        else:  # par (Q2,Q4,Q6,Q8,Q10)
            suma += 5 - val
    return suma * 2.5

df['score_calculado'] = df.apply(calcular_score_sus, axis=1)
df['score_match'] = np.isclose(df['score'], df['score_calculado'])

mismatches = df[~df['score_match']]
if len(mismatches) > 0:
    print(f'ADVERTENCIA: {len(mismatches)} scores no coinciden:')
    print(mismatches[['codigo', 'score', 'score_calculado']])
else:
    print('Todos los scores calculados coinciden con el CSV.')

df[['codigo', 'score', 'score_calculado']].style.hide(axis='index')

Todos los scores calculados coinciden con el CSV.


codigo,score,score_calculado
P01,80.000000,80.000000
P02,82.500000,82.500000
P03,85.000000,85.000000
P04,77.500000,77.500000
P05,87.500000,87.500000
P06,80.000000,80.000000
P07,82.500000,82.500000
P08,85.000000,85.000000
P09,80.000000,80.000000
P10,77.500000,77.500000


## 3. Estadística descriptiva

In [ ]:
scores = df['score_calculado']
n = len(scores)

media = scores.mean()
mediana = scores.median()
dt = scores.std(ddof=1)
minimo = scores.min()
maximo = scores.max()
p25 = scores.quantile(0.25)
p50 = scores.quantile(0.50)
p75 = scores.quantile(0.75)
p95 = scores.quantile(0.95)

print('=' * 50)
print(f'ESTADÍSTICA DESCRIPTIVA SUS (N={n})')
print('=' * 50)
print(f'Media:              {media:.2f}')
print(f'Mediana:            {mediana:.2f}')
print(f'Desviación Típica:  {dt:.2f}')
print(f'Mínimo:             {minimo:.2f}')
print(f'Máximo:             {maximo:.2f}')
print(f'Percentil 25:       {p25:.2f}')
print(f'Percentil 50:       {p50:.2f}')
print(f'Percentil 75:       {p75:.2f}')
print(f'Percentil 95:       {p95:.2f}')

## 4. Intervalo de Confianza al 95% (IC 95%)

Se utiliza la distribución *t* de Student porque la muestra mock es pequeña (bajo el umbral n<30; dataset retirado, N=0 en el entregable).

In [ ]:
ic_95 = stats.t.interval(0.95, df=n-1, loc=media, scale=stats.sem(scores))

print(f'Intervalo de Confianza al 95%: [{ic_95[0]:.2f}, {ic_95[1]:.2f}]')
print(f'Error estándar: {stats.sem(scores):.2f}')
print(f'Grados de libertad: {n-1}')

## 5. Análisis de ítems SUS (Q1–Q10)

Media y desviación estándar de cada ítem en escala Likert 1–5.

In [6]:
items_stats = df[ITEM_COLS].agg(['mean', 'std']).T
items_stats.columns = ['Media', 'DT']
items_stats['Tipo'] = ['Positivo' if i % 2 == 0 else 'Negativo' for i in range(10)]

print('Estadística por ítem SUS:')
print(items_stats.round(3).to_string())

Estadística por ítem SUS:
     Media     DT      Tipo
Q1   4.533  0.640  Positivo
Q2   1.200  0.414  Negativo
Q3   4.067  0.704  Positivo
Q4   1.067  0.258  Negativo
Q5   4.000  0.535  Positivo
Q6   1.533  0.516  Negativo
Q7   4.067  0.258  Positivo
Q8   2.000  0.000  Negativo
Q9   4.000  0.000  Positivo
Q10  2.000  0.000  Negativo


## 6. Análisis de preguntas de interfaz (I1, I2)

Preguntas adicionales de interfaz (no parte del cálculo SUS estándar):

In [7]:
i1_media = df['I1'].mean()
i1_dt = df['I1'].std(ddof=1)
i2_media = df['I2'].mean()
i2_dt = df['I2'].std(ddof=1)

print(f'I1 (Fácil de aprender):    Media={i1_media:.2f}, DT={i1_dt:.2f}')
print(f'I2 (Uso independiente):    Media={i2_media:.2f}, DT={i2_dt:.2f}')

I1 (Fácil de aprender):    Media=4.07, DT=0.70
I2 (Uso independiente):    Media=4.40, DT=0.51


## 7. Gráfico: Boxplot de puntajes SUS

In [8]:
fig, ax = plt.subplots(figsize=(6, 8))
bp = ax.boxplot(scores, vert=True, widths=0.3, patch_artist=True,
                boxprops=dict(facecolor='#2b5c8f', alpha=0.7),
                medianprops=dict(color='white', linewidth=2),
                whiskerprops=dict(color='#2b5c8f'),
                capprops=dict(color='#2b5c8f'),
                flierprops=dict(markerfacecolor='#2b5c8f', marker='o', markersize=5))

ax.axhline(68, color='#D55E00', linestyle='--', linewidth=1.5, label='Umbral aceptabilidad (68)')
ax.axhline(media, color='#009E73', linestyle='-', linewidth=1, alpha=0.5, label=f'Media ({media:.1f})')

ax.set_ylabel('Puntaje SUS (0–100)', fontsize=11)
ax.set_title(f'Distribución de Puntuaciones SUS — datos mock\n(N={n}, Media={media:.1f}, DT={dt:.1f})',
             fontsize=12, fontweight='bold')
ax.set_xticklabels(['SUS Score'])
ax.set_ylim(50, 100)
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sus_boxplot.svg', format='svg', bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'sus_boxplot.png', dpi=300, bbox_inches='tight')
plt.close()
print('Gráfico guardado: sus_boxplot.svg, sus_boxplot.png')

Gráfico guardado: sus_boxplot.svg, sus_boxplot.png


## 8. Gráfico: Desglose por ítem (Q1–Q10)

In [9]:
item_means = df[ITEM_COLS].mean()
item_stds = df[ITEM_COLS].std(ddof=1)

fig, ax = plt.subplots(figsize=(9, 5))
y_pos = np.arange(10)
colors = ['#009E73' if i % 2 == 0 else '#D55E00' for i in range(10)]

bars = ax.barh(y_pos, item_means.values, xerr=item_stds.values,
               capsize=4, color=colors, alpha=0.8, height=0.6)

ax.set_yticks(y_pos)
ax.set_yticklabels([f'Q{i+1}' + (' (+)' if i % 2 == 0 else ' (-)') for i in range(10)])
ax.set_xlabel('Media Likert (1–5)', fontsize=11)
ax.set_title('Promedio por Ítem SUS (con desviación estándar)', fontsize=12, fontweight='bold')
ax.set_xlim(0, 5.5)
ax.axvline(3, color='gray', linestyle=':', alpha=0.5, label='Neutral (3)')
ax.legend(fontsize=9)

for i, (v, s) in enumerate(zip(item_means.values, item_stds.values)):
    ax.text(v + s + 0.1, i, f'{v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sus_items_breakdown.svg', format='svg', bbox_inches='tight')
plt.close()
print('Gráfico guardado: sus_items_breakdown.svg')

Gráfico guardado: sus_items_breakdown.svg


## 9. Interpretación cualitativa (Bangor et al. 2008 / Sauro 2011)

| Rango SUS | Adjetivo | Clasificación |
|-----------|----------|---------------|
| 0–25 | Peor imaginable | No aceptable |
| 25–50 | Pobre | No aceptable |
| 50–70 | OK | Aceptable marginal |
| 70–80 | Bueno | Aceptable |
| 80–90 | Excelente | Bueno |
| 90–100 | Mejor imaginable | Excelente |

In [ ]:
def clasificacion_bangor(score):
    if score >= 90: return 'Mejor imaginable'
    if score >= 80: return 'Excelente'
    if score >= 70: return 'Bueno'
    if score >= 50: return 'OK (Aceptable)'
    if score >= 25: return 'Pobre'
    return 'Peor imaginable'

clasificacion = clasificacion_bangor(media)

print('=' * 50)
print('INTERPRETACIÓN SUS')
print('=' * 50)
print(f'Media SUS:          {media:.2f}')
print(f'IC 95%:             [{ic_95[0]:.2f}, {ic_95[1]:.2f}]')
print(f'Clasificación:      {clasificacion}')
print()
if media >= 68:
    print(f'El puntaje SUS promedio ({media:.1f}) supera el umbral de aceptabilidad (68),')
    print(f'lo que indica que el sistema SGB-SaaS tiene una usabilidad {clasificacion.lower()}.')
else:
    print(f'El puntaje SUS promedio ({media:.1f}) está por debajo del umbral (68).')

print()
print('Distribución demográfica:')
print(df[['edad', 'sexo', 'experiencia_web', 'dispositivo']].value_counts().to_string())

## 10. Amenazas a la validez

- **Muestra por conveniencia**: participantes reclutados del entorno cercano del equipo.
- **Sesgo de complacencia**: podrían puntuar más alto por conocer al equipo evaluador.
- **Entorno controlado**: sesiones supervisadas, no refleja uso orgánico/desatendido.
- **Muestra mock retirada**: validación del pipeline sin valor evidencial (N=0 en el entregable, ver OBS-08).